In [1]:
import csv
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DATA_FILE = "IMDB Dataset.csv"

POSITIVE_KEYWORDS = ["excellent", "amazing", "wonderful", "brilliant"]
NEGATIVE_KEYWORDS = ["terrible", "boring", "awful", "worst"]
KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS

In [ ]:
PUNCTUATION = ".,!?;:\"'()[]{}-/\\*&<>_=+#@$%^~`|"
CLEAN_TABLE = str.maketrans({ch: " " for ch in PUNCTUATION})

In [ ]:
def tokenize(text):
    """Lowercase, remove HTML breaks and punctuation, return the SET of words.
    Using a set means a keyword counts at most once per review."""
    text = text.lower().replace("<br />", " ")
    return set(text.translate(CLEAN_TABLE).split())


def load_data(path, keywords):
    """Read the CSV once. Returns:
       is_positive: bool array, shape (n_reviews,)
       has_keyword: dict keyword -> bool array, shape (n_reviews,)"""
    positive_flags = []
    found = {kw: [] for kw in keywords}

    with open(path, "r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        for row in reader:
            if not row.get("review") or not row.get("sentiment"):
                continue  # skip incomplete rows
            words = tokenize(row["review"])
            positive_flags.append(row["sentiment"].strip().lower() == "positive")
            for kw in keywords:
                found[kw].append(kw in words)

    is_positive = np.array(positive_flags, dtype=bool)
    has_keyword = {kw: np.array(found[kw], dtype=bool) for kw in keywords}
    return is_positive, has_keyword


def bayes_theorem(prior, likelihood, marginal):
    """P(Positive | keyword) = P(keyword | Positive) * P(Positive) / P(keyword)"""
    if marginal == 0:
        return 0.0
    return likelihood * prior / marginal


def analyze_keyword(keyword, is_positive, has_keyword_arr):
    """Compute prior, likelihood, marginal and posterior for one keyword."""
    total = len(is_positive)
    total_pos = int(np.sum(is_positive))
    kw_total = int(np.sum(has_keyword_arr))
    kw_pos = int(np.sum(has_keyword_arr & is_positive))

    prior = total_pos / total                  # P(Positive)
    likelihood = kw_pos / total_pos            # P(keyword | Positive)
    marginal = kw_total / total                # P(keyword)
    posterior = bayes_theorem(prior, likelihood, marginal)
    direct = kw_pos / kw_total if kw_total else 0.0   # sanity check

    return {
        "keyword": keyword, "prior": prior, "likelihood": likelihood,
        "marginal": marginal, "posterior": posterior, "direct": direct,
        "total": kw_total, "positive": kw_pos,
    }


def print_table(r):
    """Print one markdown table per keyword."""
    print(f"### Keyword: \"{r['keyword']}\"  ({r['total']} reviews, "
          f"{r['positive']} positive)")
    print("| Probability                         | Value  |")
    print("|-------------------------------------|--------|")
    print(f"| Prior: P(Positive)                  | {r['prior']:.4f} |")
    print(f"| Likelihood: P(keyword | Positive)   | {r['likelihood']:.4f} |")
    print(f"| Marginal: P(keyword)                | {r['marginal']:.4f} |")
    print(f"| Posterior: P(Positive | keyword)    | {r['posterior']:.4f} |")
    print(f"| Check: direct count                 | {r['direct']:.4f} |")
    print()


def make_graphs(results):
    names = [r["keyword"] for r in results]
    x = np.arange(len(names))

    # Graph 1: posterior per keyword (green = positive keyword, red = negative)
    posterior = np.array([r["posterior"] for r in results]) * 100
    colors = ["tab:green" if n in POSITIVE_KEYWORDS else "tab:red" for n in names]
    plt.figure(figsize=(9, 5))
    bars = plt.bar(names, posterior, color=colors)
    plt.axhline(50, color="gray", linestyle="--", label="50% (neutral)")
    plt.title("Posterior Probability P(Positive | keyword)")
    plt.xlabel("Keyword")
    plt.ylabel("Probability (%)")
    plt.ylim(0, 100)
    plt.bar_label(bars, fmt="%.1f%%", padding=3)
    plt.legend()
    plt.tight_layout()
    plt.savefig("graph1_posterior.png", dpi=150)

    # Graph 2: prior vs likelihood vs posterior (marginal is tiny, so it
    # is shown in Graph 3 on its own scale)
    plt.figure(figsize=(10, 5))
    width = 0.25
    for i, (key, label) in enumerate([("prior", "Prior"),
                                      ("likelihood", "Likelihood"),
                                      ("posterior", "Posterior")]):
        values = np.array([r[key] for r in results]) * 100
        plt.bar(x + (i - 1) * width, values, width=width, label=label)
    plt.title("Prior, Likelihood and Posterior by Keyword")
    plt.xlabel("Keyword")
    plt.ylabel("Probability (%)")
    plt.xticks(x, names)
    plt.legend()
    plt.tight_layout()
    plt.savefig("graph2_probabilities.png", dpi=150)

    # Graph 3: review counts
    totals = [r["total"] for r in results]
    positives = [r["positive"] for r in results]
    width = 0.38
    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, totals, width=width, label="All reviews with keyword")
    plt.bar(x + width / 2, positives, width=width,
            label="Positive reviews with keyword")
    plt.title("Review Counts by Keyword")
    plt.xlabel("Keyword")
    plt.ylabel("Number of reviews")
    plt.xticks(x, names)
    plt.legend()
    plt.tight_layout()
    plt.savefig("graph3_counts.png", dpi=150)


def main():
    is_positive, has_keyword = load_data(DATA_FILE, KEYWORDS)
    print(f"Total reviews:    {len(is_positive)}")
    print(f"Positive reviews: {int(np.sum(is_positive))}\n")

    results = []
    for kw in KEYWORDS:
        r = analyze_keyword(kw, is_positive, has_keyword[kw])
        results.append(r)
        print_table(r)

    make_graphs(results)
    print("Graphs saved as PNG files. Close each window to continue.")
    plt.show()


if __name__ == "__main__":
    main()